BGWO's: RF
BGWO optimizes the hyperparameters of the Random Forest classifier. The fitness function evaluates the classifier's accuracy on the test set, guiding BGWO to find the best hyperparameters.

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
X = pd.read_excel('minmax.xlsx')
y = pd.read_csv('idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for BGWO
def fitness_function(position):
    n_estimators = int(position[0])
    max_depth = int(position[1])
    
    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize BGWO parameters
num_wolves = 10
num_features = 2  # n_estimators and max_depth
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly
wolves = np.random.randint(low=[10, 1], high=[200, 20], size=(num_wolves, num_features))

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3
            
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            
            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta
            
            wolves[i, j] = (X1 + X2 + X3) / 3

# Evaluate the best solution
best_wolf = alpha
best_n_estimators = int(best_wolf[0])
best_max_depth = int(best_wolf[1])

final_model = RandomForestClassifier(n_estimators=best_n_estimators, max_depth=best_max_depth, random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Print performance metrics
print(f"Best Hyperparameters: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

C:\Users\a2004\AppData\Local\Temp\ipykernel_12804\2408417300.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Best Hyperparameters: n_estimators=195, max_depth=10
Accuracy: 95.51%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2
           2       1.00      1.00      1.00         8
           3       1.00      0.83      0.91         6
           4       0.94      0.94      0.94        18
           5       0.90      1.00      0.95         9
           6       1.00      1.00      1.00         6
           7       1.00      1.00      1.00         1
           9       0.93      1.00      0.96        13
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         5
          12       0.86      1.00      0.92         6
          13       1.00      0.71      0.83         7
          14       1.00      1.00      1.00         7

    accuracy                           0.96        89
   macro avg       0.97      0.96      0.96        89
weighted avg       0.96      0.96      0

BGWO's: SVM
BGWO optimizes the hyperparameters of the SVM classifier. The fitness function evaluates the classifier's accuracy on the test set, guiding BGWO to find the best hyperparameters.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC

# Load the dataset
X = pd.read_excel('minmax.xlsx')
y = pd.read_csv('idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for BGWO
def fitness_function(position):
    C = 10 ** position[0]  # Map to a range of 10^x for C
    gamma = 10 ** position[1]  # Map to a range of 10^x for gamma
    
    # Train an SVM model
    model = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize BGWO parameters
num_wolves = 10
num_features = 2  # C and gamma
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (log10 scale for C and gamma)
wolves = np.random.uniform(low=[-3, -3], high=[3, 3], size=(num_wolves, num_features))

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3
            
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            
            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta
            
            wolves[i, j] = (X1 + X2 + X3) / 3

# Evaluate the best solution
best_wolf = alpha
best_C = 10 ** best_wolf[0]
best_gamma = 10 ** best_wolf[1]

final_model = SVC(C=best_C, gamma=best_gamma, kernel='rbf', random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

# Print performance metrics
print(f"Best Hyperparameters: C={best_C:.6f}, gamma={best_gamma:.6f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Best Hyperparameters: C=3.926567, gamma=0.005810
Accuracy: 97.75%

Classification Report:
              precision    recall  f1-score   support

           1       1.00      1.00      1.00         2
           2       1.00      1.00      1.00         8
           3       0.86      1.00      0.92         6
           4       1.00      0.94      0.97        18
           5       0.90      1.00      0.95         9
           6       1.00      0.83      0.91         6
           7       1.00      1.00      1.00         1
           9       1.00      1.00      1.00        13
          10       1.00      1.00      1.00         1
          11       1.00      1.00      1.00         5
          12       1.00      1.00      1.00         6
          13       1.00      1.00      1.00         7
          14       1.00      1.00      1.00         7

    accuracy                           0.98        89
   macro avg       0.98      0.98      0.98        89
weighted avg       0.98      0.98      0.98 

feature + hyper

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# Load the dataset
X = pd.read_excel('minmax.xlsx')  # Midinfrared data
y = pd.read_csv('idC_with_header.csv')['Label']  # Classes

# Step 1: Apply PCA for dimensionality reduction
n_components = 50  # Reduce to 50 principal components
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X)

# Step 2: Split the PCA-transformed data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# Step 3: Define the fitness function for BGWO
def fitness_function(position):
    # Decode the binary vector for feature selection
    selected_features = np.where(position[:n_components] > 0.5)[0]
    if len(selected_features) == 0:  # Avoid empty feature selection
        return 0

    # Extract selected features
    X_train_selected = X_train[:, selected_features]
    X_test_selected = X_test[:, selected_features]

    # Extract hyperparameters from the position vector
    n_estimators = int(position[n_components])  # Number of trees
    max_depth = int(position[n_components + 1])  # Maximum depth

    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train_selected, y_train)

    # Evaluate the model on the test set
    y_pred = model.predict(X_test_selected)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Step 4: Initialize BGWO parameters
num_wolves = 10
num_features = n_components + 2  # Binary vector for features + 2 hyperparameters
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly
wolves = np.random.rand(num_wolves, num_features)
wolves[:, n_components:] = np.random.randint(low=[10, 1], high=[200, 20], size=(num_wolves, 2))  # Hyperparameters

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])

    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]

    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            r1, r2, r3 = np.random.rand(3)
            A1, A2, A3 = 2 * r1 - 1, 2 * r2 - 1, 2 * r3 - 1
            C1, C2, C3 = 2 * r1, 2 * r2, 2 * r3

            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            D_delta = abs(C3 * delta[j] - wolves[i, j])

            X1 = alpha[j] - A1 * D_alpha
            X2 = beta[j] - A2 * D_beta
            X3 = delta[j] - A3 * D_delta

            wolves[i, j] = (X1 + X2 + X3) / 3

# Step 5: Evaluate the best solution
best_wolf = alpha
selected_features = np.where(best_wolf[:n_components] > 0.5)[0]
best_n_estimators = int(best_wolf[n_components])
best_max_depth = int(best_wolf[n_components + 1])

X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

final_model = RandomForestClassifier(n_estimators=best_n_estimators, max_depth=best_max_depth, random_state=42)
final_model.fit(X_train_selected, y_train)
y_pred = final_model.predict(X_test_selected)

# Print performance metrics
print(f"Selected Features: {len(selected_features)} out of {n_components}")
print(f"Best Hyperparameters: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))